In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from python_module.pricing_model import BSMModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.4f}'.format

In [2]:
symbol = 'QQQ'
exp_window = 20
target_vol = 0.10
strike_delta = -0.1
day_to_maturity = 10
slide_scenario = -0.3

In [3]:
price_df = pd.read_csv(f'data/{symbol}.csv', index_col=0, parse_dates=True)['price'].to_frame(symbol)

In [4]:
init_vt_price = 100
log_returns_df = np.log(price_df).diff()
rolling_std_df = log_returns_df.ewm(span=exp_window, min_periods=exp_window, adjust=False).std()
rolling_std_df *= (252 ** 0.5)
leverage_df = target_vol / rolling_std_df
vt_price_df = (price_df.loc[leverage_df.dropna().index].pct_change() * leverage_df.shift(1)).fillna(0).add(1).cumprod() * init_vt_price
vt_price_df.columns = ['F']
agg_df = pd.concat([price_df, leverage_df[symbol].to_frame('beta'), vt_price_df], axis=1).dropna()
agg_df['vt_delta'] = (agg_df['F'] * agg_df['beta'])/agg_df[symbol]

In [5]:
def apply_compute_option(row):
    inputs = row[['F', 'K', 'T', 'r', 'sigma', 'option_type', 'compute_greeks', 'slide_scenario', 'slide_compute']].to_dict()
    return BSMModel.compute_option(**inputs)

i = 0
results = list()
for date in tqdm(agg_df.index):
    i += 1
    agg_df_temp = agg_df.loc[date:].iloc[:day_to_maturity+1]
    if agg_df_temp.shape[0] != day_to_maturity+1: break
    agg_df_temp['strike_date'] = agg_df_temp.index[0]
    agg_df_temp['maturity_date'] = agg_df_temp.index[-1]
    agg_df_temp['days_to_maturity'] = list(range(agg_df_temp.shape[0]))[::-1]
    agg_df_temp['T'] = agg_df_temp['days_to_maturity'] / 252

    option_type = 'call' if strike_delta > 0 else 'put'
    agg_df_temp['option_type'] = option_type

    strike_k = BSMModel.solve_delta_strike(F=init_vt_price, T=day_to_maturity/252, sigma=target_vol, r=0, option_type=option_type, target_delta=strike_delta)
    strike_pct = strike_k / init_vt_price
    agg_df_temp['K'] = agg_df_temp['F'].iloc[0] * strike_pct
    agg_df_temp['r'] = 0
    agg_df_temp['sigma'] = target_vol
    agg_df_temp['compute_greeks'] = True
    agg_df_temp['slide_scenario'] = slide_scenario
    agg_df_temp['slide_compute'] = 'option_pnl'

    pricing_df = agg_df_temp.apply(lambda x: apply_compute_option(x), axis=1, result_type='expand')
    agg_df_temp = pd.concat([agg_df_temp, pricing_df], axis=1)
    agg_df_temp['asset_delta'] = agg_df_temp['vt_delta'] * agg_df_temp['delta']
    agg_df_temp['asset_delta_cash'] = agg_df_temp[symbol] * agg_df_temp['asset_delta']
    agg_df_temp['dP'] = agg_df_temp['price'].diff()
    agg_df_temp['dH'] = agg_df_temp[symbol].diff() * agg_df_temp['asset_delta'].shift(1)
    results.append(agg_df_temp)

100%|█████████▉| 6487/6497 [01:03<00:00, 102.55it/s]


In [90]:
results_df = pd.concat(results)
trading_units = 20
trading_scale = 'theta' # 'F', 'price', 'delta', 'gamma', 'vega', 'theta', 'vanna', 'volga', slide_scenario
scaling_factor = trading_units / results_df.groupby('strike_date')[trading_scale].first()
results_df['scaling_factor'] = results_df['strike_date'].map(scaling_factor)

In [91]:
for col in ['asset_delta_cash','dP', 'dH']:
    results_df[f'scaled_{col}'] = results_df['scaling_factor'] * results_df[col]
results_df[f'scaled_{trading_scale}'] = results_df['scaling_factor'] * results_df[trading_scale]
results_df[f'scaled_{slide_scenario}'] = results_df['scaling_factor'] * results_df[slide_scenario]
nb_strike = results_df.groupby(results_df.index)['strike_date'].count()
mask = (nb_strike == (day_to_maturity+1))
results_df = results_df.loc[mask]

In [92]:
results_df.groupby(results_df.index)['scaled_asset_delta_cash'].sum()

2000-02-15     9,140.0526
2000-02-16    10,655.3268
2000-02-17     7,373.3933
2000-02-18    15,374.4305
2000-02-22    14,204.3976
                 ...     
2025-11-07   143,816.6403
2025-11-10    30,016.4846
2025-11-11    31,244.0485
2025-11-12    29,448.9443
2025-11-13    83,791.7503
Name: scaled_asset_delta_cash, Length: 6477, dtype: float64

In [93]:
px.line(results_df.groupby(results_df.index)['scaled_asset_delta_cash'].sum())

In [94]:
daily_pnl = results_df.groupby(results_df.index)['scaled_dH'].sum()
yearly_pnl = daily_pnl.groupby(daily_pnl.index.year).sum().diff()
cum_pnl = daily_pnl.cumsum()

In [95]:
yearly_pnl.describe()

count        25.0000
mean        226.9330
std      18,204.2104
min     -47,507.4767
25%     -10,929.4272
50%         646.2123
75%      14,232.0807
max      34,301.4034
Name: scaled_dH, dtype: float64

In [96]:
yearly_pnl

2000            NaN
2001   -20,492.8174
2002    23,875.1126
2003     5,420.2779
2004   -15,196.8287
2005    -3,434.3304
2006     2,999.8292
2007       646.2123
2008   -10,929.4272
2009    30,972.5640
2010   -13,251.1664
2011    -8,491.8260
2012    15,753.2846
2013     7,927.5868
2014    -2,055.5764
2015   -15,127.6111
2016      -862.0788
2017    34,301.4034
2018   -47,507.4767
2019    15,314.7421
2020    -4,884.4085
2021    14,232.0807
2022   -24,828.2801
2023    17,053.4391
2024     1,348.2100
2025     2,890.4105
Name: scaled_dH, dtype: float64

In [74]:
px.line(results_df.groupby(results_df.index)['scaled_dH'].sum().cumsum())